In [ ]:
pip install pandas pyarrow torch transformers tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from datetime import datetime
import os
import warnings

# Suppress Hugging Face warnings (e.g., about tokenizers)
warnings.filterwarnings("ignore", category=UserWarning)

# ===============================================================
# CONFIGURATION (CLEANED AND CORRECTED)
# ===============================================================
# The CWD is already '/lustre/home/skylar2/Sentiment_Analys'.
# The file is located directly in this directory.

# --- PATH CORRECTION ---
# 1. Define the exact filename
INPUT_FILE_NAME = "KOSPI_RawNews_Classified_v2_20251209.parquet"

# 2. Set the INPUT_FILE_PATH directly to the filename (searches in CWD).
INPUT_FILE_PATH = INPUT_FILE_NAME
# -----------------------

# Define the directory where both input and output files reside (CWD)
# This variable was missing and caused the NameError.
INPUT_DIR = "."

# Set the OUTPUT_DIR using the corrected INPUT_DIR
OUTPUT_DIR = INPUT_DIR  # '.' denotes the current directory
OUTPUT_FILENAME_PREFIX = "KOSPI_Sentiment_Analyzed"
MODEL_NAME = "snunlp/KR-FinBert-SC"


# ===============================================================
# SENTIMENT ANALYSIS FUNCTION (Unchanged)
# ===============================================================
def analyze_sentiment_batch(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyzes the sentiment of the news titles in the DataFrame using
    the specified Korean FinBert model.
    """
    # Determine the device for PyTorch
    device_index = 0 if torch.cuda.is_available() else -1
    print(f"Loading sentiment model ({MODEL_NAME}) on device: {'GPU' if device_index != -1 else 'CPU'}")

    try:
        # Load the model and tokenizer explicitly to handle potential issues
        print("➡️ 1/3: Loading Tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

        print("➡️ 2/3: Loading Model Weights...")
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

        print("➡️ 3/3: Initializing Classification Pipeline...")

        # Determine data type for memory efficiency on GPU
        dtype = torch.float32
        if device_index != -1:
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

        # Create the sentiment analysis pipeline
        classifier = pipeline(
            "sentiment-analysis",
            model=model,
            tokenizer=tokenizer,
            device=device_index,
            truncation=True,
            max_length=512,
            batch_size=8,
            torch_dtype=dtype
        )
        print("✅ Model Loading Complete.")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Please ensure 'transformers' and 'torch' are correctly installed.")
        df["sentiment"] = None
        df["sentiment_score"] = None
        return df

    # Prepare texts.
    texts = df["title"].astype(str).fillna("").tolist()

    if not texts:
        print("No texts found for analysis.")
        return df

    print(f"\nStarting sentiment analysis for {len(texts)} articles...")

    # Run the classification
    try:
        results = list(tqdm(classifier(texts), desc="Sentiment Analysis", total=len(texts)))

        # Extract results
        df["sentiment"] = [r["label"] for r in results]
        df["sentiment_score"] = [r["score"] for r in results]
    except Exception as e:
        print(f"❌ Error during classification: {e}")
        df["sentiment"] = None
        df["sentiment_score"] = None

    return df

# ===============================================================
# MAIN EXECUTION
# ===============================================================
if __name__ == "__main__":
    print(f"Attempting to load data from: {INPUT_FILE_PATH}")

    # NOTE: The os.path.exists check is critical for file paths.
    if not os.path.exists(INPUT_FILE_PATH):
        print(f"Error: Input file not found at '{INPUT_FILE_PATH}'")
        # Added helpful path diagnostic
        print("\n--- Diagnostic Info ---")
        print(f"Current Working Directory (CWD): {os.getcwd()}")
        print(f"Expected absolute path: {os.path.abspath(INPUT_FILE_PATH)}")
        print(f"Please confirm the file '{INPUT_FILE_NAME}' exists directly in your CWD.")
        print("-----------------------")
        exit()

    try:
        df_raw = pd.read_parquet(INPUT_FILE_PATH)
        print(f"Successfully loaded {len(df_raw)} articles.")

        df_analyzed = analyze_sentiment_batch(df_raw)

        # 2. Save the result
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        out_name = os.path.join(OUTPUT_DIR, f"{OUTPUT_FILENAME_PREFIX}_{timestamp}.parquet")

        # Ensure output directory exists (The same as the input directory)
        if not os.path.exists(OUTPUT_DIR):
            os.makedirs(OUTPUT_DIR)

        df_analyzed.to_parquet(out_name, index=False)

        print("\n====================================")
        print("SENTIMENT ANALYSIS COMPLETED")
        print("Saved results to:", out_name)
        print("Total articles analyzed:", len(df_analyzed))
        print("New columns added: 'sentiment', 'sentiment_score'")
        print("====================================")

        print("Sample Output (New Columns Highlighted):")
        print(df_analyzed[["company", "published_at", "title", "sentiment", "sentiment_score"]].head())

    except Exception as e:
        print(f"An unexpected error occurred during the main execution: {e}")

Attempting to load data from: KOSPI_RawNews_Classified_v2_20251209.parquet
Successfully loaded 795648 articles.
Loading sentiment model (snunlp/KR-FinBert-SC) on device: GPU
➡️ 1/3: Loading Tokenizer...
➡️ 2/3: Loading Model Weights...


Device set to use cuda:0


➡️ 3/3: Initializing Classification Pipeline...
✅ Model Loading Complete.

Starting sentiment analysis for 795648 articles...


Sentiment Analysis: 100%|██████████████████████████████████████████████████████████████████████████████████████| 795648/795648 [00:00<00:00, 4904869.15it/s]



SENTIMENT ANALYSIS COMPLETED
Saved results to: ./KOSPI_Sentiment_Analyzed_20251209_154638.parquet
Total articles analyzed: 795648
New columns added: 'sentiment', 'sentiment_score'
Sample Output (New Columns Highlighted):
  company                   published_at  \
0    삼성전자  2023-01-30T00:00:00.000+09:00   
1    삼성전자  2023-01-30T00:00:00.000+09:00   
2    삼성전자  2023-01-30T00:00:00.000+09:00   
3    삼성전자  2023-01-30T00:00:00.000+09:00   
4    삼성전자  2023-01-30T00:00:00.000+09:00   

                                           title sentiment  sentiment_score  
0  조나단 무어 PKF 재무부문 대표 "美 투자자, 중국 비중 줄이고 한국 늘릴 것"   neutral         0.997314  
1             ​삼성중공업, 작년 영업손실 8544억…"올해 흑자전환 전망"  negative         0.982866  
2                          삼성 13연패 제물로…캐롯은 연패 탈출  negative         0.681420  
3                  31일부터 ‘봄배구’ 향해 레이스…3위 싸움 ‘불꽃’   neutral         0.999903  
4                가을 수확은 지금부터… 10개 구단 ‘숙제’ 풀러 나간다   neutral         0.999910  
